在数据预处理中，**编码（Encoding）**的任务是将非数值型的分类变量（如“城市”、“颜色”、“等级”）转化为机器学习模型能够理解的数值向量。

这是建模中处理**定性指标（Categorical Data）**的必经之路。

---

### 一、 标签编码 (Label Encoding)

#### 1. 算法原理与数学推导
**核心思想**：通过一个双射函数（Bijective Mapping），将 $n$ 个类别映射到 $\{0, 1, 2, \dots, n-1\}$ 的整数空间。

**数学表达**：
设分类集合为 $C = \{c_1, c_2, \dots, c_n\}$，映射函数 $f: C \to \mathbb{N}$：
$$ f(c_i) = i - 1 $$
例如：`['小学', '初中', '高中']` 映射为 `[0, 1, 2]`。

**数学逻辑与缺陷**：
标签编码引入了**大小关系**和**距离度量**。在数学上：
$$ 2 > 1 > 0 \implies \text{高中} > \text{初中} > \text{小学} $$
$$ |2 - 1| = |1 - 0| \implies \text{高中与初中的距离} = \text{初中与小学的距离} $$
*   **优点**：如果类别本身具有顺序性（等级、程度），这种映射能很好地保留其序数信息。
*   **缺点**：如果类别是平等的（如“北京”、“上海”），标签编码会错误地赋予它们本不存在的权重和距离，导致线性模型（回归、神经网络）产生偏见。

#### 2. Python 代码
```python
from sklearn.preprocessing import LabelEncoder

data = ['小学', '高中', '初中', '高中', '小学']
le = LabelEncoder()
encoded_data = le.fit_transform(data)
print(f"标签编码结果: {encoded_data}")
# 输出: [0 2 1 2 0]
```

---

### 二、 独热编码 (One-Hot Encoding)

#### 1. 算法原理与数学推导
**核心思想**：将每个类别扩展为一个高维空间的**标准基向量（Standard Basis Vector）**，消除人为引入的顺序性。

**数学推导**：
假设类别数为 $n$，我们将每一个类别映射到一个 $n$ 维的欧式空间 $\mathbb{R}^n$。
第 $i$ 个类别映射为单位向量 $e_i$，即只有第 $i$ 位为 1，其余为 0：
*   $c_1 \to [1, 0, 0, \dots, 0]$
*   $c_2 \to [0, 1, 0, \dots, 0]$

**数学逻辑（距离特性）**：
在独热编码空间中，任意两个不同类别向量 $e_i, e_j$ 之间的欧氏距离恒等于 $\sqrt{2}$：
$$ ||e_i - e_j|| = \sqrt{(1-0)^2 + (0-1)^2 + 0} = \sqrt{2} $$
**推导结论**：
独热编码使得所有类别在几何空间上是**完全等距**的。这消除了标签编码中“高中比初中近，比小学远”的数值偏见，确保了分类的公平性。

#### 2. Python 代码
```python
import pandas as pd

df = pd.DataFrame({'城市': ['北京', '上海', '广州', '北京']})
# 使用 pandas 的 get_dummies 极其方便
one_hot = pd.get_dummies(df['城市'])
print(one_hot)
```

---

### 三、 建模实战：论文中的“修修补补”与陷阱

在写数学建模论文时，关于编码的选择可以体现出你对模型底层逻辑的理解：

#### 1. 模型对编码的偏好（加分项）：
*   **线性模型（逻辑回归、SVM、神经网络）**：必须使用**独热编码**。因为这些模型依赖距离或权重的线性组合，如果使用标签编码，模型会把“城市=2”理解为“城市=1”的两倍。
*   **树模型（决策树、随机森林、XGBoost）**：通常使用**标签编码**即可。树模型是通过寻找最佳切分点（Split Point）来分支的，它不计算距离，因此标签编码的数值大小不会干扰它的分裂逻辑。

#### 2. 虚拟变量陷阱 (Dummy Variable Trap) —— 数学推导
在做**多元线性回归**时，如果你有 $n$ 个类别，且引入了 $n$ 列独热编码，同时模型里还有**截距项（Intercept）**，就会产生**完全多重共线性**。
**数学解释**：
因为 $x_{onehot\_1} + x_{onehot\_2} + \dots + x_{onehot\_n} = 1$（恒等于常数项）。
这导致矩阵 $X^T X$ 不可逆，模型崩溃。
*   **修正**：在回归分析中，永远只保留 $n-1$ 列独热编码（即 `drop_first=True`）。

#### 3. 维度灾难：
如果一个特征有 1000 个城市名，独热编码会瞬间增加 1000 列，导致计算爆炸。
*   **修补建议**：在论文中写道：“由于‘城市’变量具有高基数特征，直接进行独热编码会导致特征空间过于稀疏。本文采用了**目标编码（Target Encoding）**或**频率编码**，以降低维度并保留类别信息。”

#### 4. 论文描述模板：
> “由于‘受教育程度’变量具有明显的定序特征（定序尺度），本文采用**标签编码**将其转化为数值序列，以保留其层级属性。而对于‘地理区域’等定类变量（定类尺度），为消除数值大小对模型权重分配的干扰，本文引入**独热编码**将其转化为 $N-1$ 维的虚拟变量，从而实现了分类特征在欧氏空间内的等距映射。”

**下一类数据预处理技术（如：PCA 主成分分析的协方差矩阵推导、等宽/等频离散化、或者是缺失值处理的 EM 算法），你想看哪一个？**